# Financial Entity Resolution & Bitemporal Ontology Crosswalk (No API Key)

Resolve Indian banks, NBFCs, co-operative banks, MCA CINs, ISO 17442 LEIs, districts, and publishers to one canonical ID through the live [LiquiLens Ontology API](https://liquilens.in/ontology/).

Key properties demonstrated in this notebook:
1. **Deterministic resolution ladder:** canonical ID -> key space (CIN, LEI, RBI, DICGC, FDIC) -> reviewed alias -> unique legal name.
2. **Never guessed:** ambiguous names (such as `Aurangabad`, a district in two Indian states) return every candidate instead of guessing.
3. **Never minted:** unknown strings return `not_found`.

- Live web lookup: https://liquilens.in/ontology/
- Bond-Market Ontology Workbench: https://liquilens.in/world-economy/bond-markets/
- OData v4 feed for Excel and Power BI: https://api.liquilens.in/api/experimental/v1/ontology/odata

**Boundary:** Identity is not evidence. An ontology ID identifies which entity a figure belongs to; it is not a credit rating or supervisory determination.

In [ ]:
import json
from urllib.parse import quote
from urllib.request import Request, urlopen

ONTO = "https://api.liquilens.in/api/experimental/v1/ontology"

def get_json(url: str) -> dict:
    req = Request(url, headers={"Accept": "application/json", "User-Agent": "LiquiLens-Colab-Ontology/1.0"})
    with urlopen(req, timeout=20) as resp:
        return json.loads(resp.read().decode("utf-8"))

census = get_json(f"{ONTO}/census")
print("Live ontology census keys:", sorted(census.keys()))
print(json.dumps({k: census.get(k) for k in ("objects", "links", "by_kind", "as_of", "schema") if k in census}, indent=2))

In [ ]:
queries = [
    "Cosmos Co-operative Bank",
    "L65910KL1997PLC011300",
    "Reserve Bank of India",
    "Aurangabad",
    "Nonexistent Lender 99999",
]

for q in queries:
    res = get_json(f"{ONTO}/resolve?q={quote(q)}")
    status = res.get("status") or res.get("resolution")
    oid = res.get("id") or (res.get("object") or {}).get("id")
    cands = len(res.get("candidates") or [])
    print(f"{q:28s} -> status={status} id={oid} candidates={cands}")

In [ ]:
# Inspect one canonical object with its cross-dataset keys and typed links
obj = get_json(f"{ONTO}/objects/source:in/self/reserve-bank-of-india")
print(json.dumps(obj, indent=2)[:1800])